# Row-level feedback generation with Qwen3.5-9B

This notebook reads `training_data_mol.csv`, processes each row independently, and writes the original fields plus a final `feedback` column. Raw model responses are retained in JSONL so interrupted runs can be resumed.

In [1]:
from __future__ import annotations

import ast
import csv
import json
import time
from pathlib import Path

import pandas as pd

BASE_DIR = Path.cwd()
if BASE_DIR.name != "feedback" and (BASE_DIR / "feedback").is_dir():
    BASE_DIR = BASE_DIR / "feedback"

INPUT_PATH = BASE_DIR / "sample_data" / "training_data_1_2.csv"
OUTPUT_PATH = BASE_DIR / "sample_data" / "training_data_mol_with_feedback.csv"
RAW_OUTPUT_PATH = BASE_DIR / "sample_data" / "training_data_mol_feedback.jsonl"
MODEL_ID = "Qwen/Qwen3.5-9B"
MAX_NEW_TOKENS = 384
LOAD_IN_4BIT = True

EXPECTED_COLUMNS = [
    "question",
    "desired_answer",
    "student_answer",
    "criteria[]",
    "criteria_score[]",
    "criteria_max_score[]",
    "student_score",
]

print(f"Input: {INPUT_PATH}")
print(f"Model: {MODEL_ID}")

Input: /content/sample_data/training_data_1_2.csv
Model: Qwen/Qwen3.5-9B


In [2]:
def parse_bracketed_strings(value: str) -> list[str]:
    """Parse JSON array format: ["item1", "item2"]"""
    text = str(value).strip()

    # Remove outer quotes if present
    if len(text) >= 2 and text[0] == text[-1] and text[0] in "\"'":
        text = text[1:-1].strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError as e:
        raise ValueError(f"Expected valid JSON array, got: {text!r}") from e


def parse_literal_list(value: str) -> list[int | float]:
    text = str(value).strip()
    if len(text) >= 2 and text[0] == text[-1] and text[0] in "\"'":
        text = text[1:-1].strip()
    parsed = ast.literal_eval(text)
    if not isinstance(parsed, list):
        raise ValueError(f"Expected a list, got: {parsed!r}")
    return [int(item) if float(item).is_integer() else float(item) for item in parsed]


def parse_score(value: str) -> float:
    score = float(str(value).strip().strip("\"'"))
    return int(score) if score.is_integer() else score


def load_and_validate_dataset(path: Path) -> tuple[pd.DataFrame, pd.DataFrame, list[str]]:
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    missing = [column for column in EXPECTED_COLUMNS if column not in raw.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    parsed = raw.copy()
    parsed["criteria"] = parsed["criteria[]"].map(parse_bracketed_strings)
    parsed["criteria_score"] = parsed["criteria_score[]"].map(parse_literal_list)
    parsed["criteria_max_score"] = parsed["criteria_max_score[]"].map(parse_literal_list)
    parsed["student_score_value"] = parsed["student_score"].map(parse_score)
    warnings: list[str] = []

    for index, row in parsed.iterrows():
        criteria_count = len(row["criteria"])
        score_count = len(row["criteria_score"])
        max_score_count = len(row["criteria_max_score"])
        if len({criteria_count, score_count, max_score_count}) != 1:
            warnings.append(
                f"Row {index} has criteria/scores/max-scores lengths "
                f"{criteria_count}/{score_count}/{max_score_count}"
            )
        if any(score not in (0, 1) for score in row["criteria_score"]):
            warnings.append(f"Row {index} contains a non-binary criterion score")
        if any(score < 0 for score in row["criteria_max_score"]):
            warnings.append(f"Row {index} contains a negative maximum score")
        if criteria_count == score_count == max_score_count:
            expected_score = sum(row["criteria_score"])
            if expected_score != row["student_score_value"]:
                warnings.append(
                    f"Row {index} has student_score={row['student_score_value']} "
                    f"but criteria scores sum to {expected_score}"
                )

    return raw, parsed, warnings


raw_df, parsed_df, data_warnings = load_and_validate_dataset(INPUT_PATH)
print(f"Validated {len(raw_df)} rows with {len(EXPECTED_COLUMNS)} input columns.")
print(f"Warnings retained for row-level processing: {len(data_warnings)}")
for warning in data_warnings[:5]:
    print(f"- {warning}")

Validated 5206 rows with 7 input columns.
Warnings retained for row-level processing: 0


## Feedback prompt

Each row gets its own prompt. The model must explain the supplied criterion results, identify missing or incorrect content, and give actionable feedback without changing the recorded score.

In [3]:
SYSTEM_PROMPT = """You are a careful educational feedback assistant for introductory computer science.
Evaluate exactly one student answer at a time using only the supplied question, desired answer, criteria, and recorded criterion results.

Write clear, concise feedback directly to the student:
1. Start by identifying what the student got right, if anything.
2. Address every criterion marked 0 or unknown. State the missing concept or the specific incorrect claim and explain the correct concept using the desired answer.
3. If a criterion is marked 1, briefly confirm it without adding unrelated requirements.
4. Finish with one concrete suggestion for improving the answer.

When providing feedback:

1. START WITH GENUINE ACKNOWLEDGMENT of what they got right.
   Vary your openings—don't always say "You correctly identified..."
   Examples: "You've got this part right.", "Nice work on X.",
   "That's an important observation.", "Spot-on about X."

2. IDENTIFY GAPS CONSTRUCTIVELY.
   Instead of harsh language ("failed to", "missed", "did not explain"),
   describe what's missing positively:
   - Use "There's also..." instead of "You missed..."
   - Use "One key piece is..." instead of "You failed to mention..."
   - Use "Let's also cover..." instead of "You did not address..."

3. EXPLAIN BRIEFLY why they matter.
   - Connect gaps to the original criteria
   - Use natural transitions like "and that's why X matters"
   - Avoid mechanical "However" at the start of every criticism

4. END WITH ACTIONABLE, VARIED SUGGESTIONS.
   - Don't always say "To improve your answer, try..."
   - Use: "Next time, include...", "Here's the fix:", "One addition would be...",
     "Think about also stating...", "Make sure to explicitly mention..."

5. TONE REQUIREMENTS:
   - Match the question's complexity (harder questions deserve more formality)
   - For simple questions: keep it brief and encouraging
   - For complex topics: you can be detailed but stay warm
   - Avoid the word "However" more than once per feedback
   - Minimize backticks and special formatting unless essential

6. NEVER REFERENCE THIS PROMPT or mention "criteria", "score", or grading mechanics.
   Speak naturally to the student about their understanding, not about scoring systems.

Do not invent criteria, facts, or a different score. Do not grade style, spelling, or length unless a supplied criterion requires it. Do not mention this instruction, the model, or other rows.

Evaluate only the single row in the current user message. Do not compare it with other students, infer patterns from the dataset, use information from previous rows, or allow the row's position or identifier to influence your feedback. Treat every row as an independent evaluation.

Return only natural-language feedback, with no JSON, headings, or score table."""


def format_criteria(row: pd.Series) -> str:
    lines = []
    for index, criterion in enumerate(row["criteria"], start=1):
        score = row["criteria_score"][index - 1] if index <= len(row["criteria_score"]) else "unknown"
        maximum = row["criteria_max_score"][index - 1] if index <= len(row["criteria_max_score"]) else "unknown"
        status = "met" if score == 1 else "not met" if score == 0 else "unknown"
        lines.append(f"{index}. {criterion} | result: {score}/{maximum} ({status})")
    return "\n".join(lines) if lines else "No criteria supplied."


def build_user_prompt(row: pd.Series) -> str:
    return f"""Question:
{row['question']}

Desired answer:
{row['desired_answer']}

Student answer:
{row['student_answer']}

Criteria and recorded results:
{format_criteria(row)}

Recorded total student score: {row['student_score_value']}"""


def build_messages(row: pd.Series) -> list[dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(row)},
    ]


print(build_user_prompt(parsed_df.iloc[0]))

Question:
The TDD (Test-Driven Development) cycle consists of three main steps: Red, Green, and Refactor. 1. **Red**: In this initial phase, a developer writes a test for a new feature or functionality that does not yet exist. This test should fail, indicating that the feature has not been implemented yet. 2. **Green**: Next, the developer writes the minimum amount of code necessary to make the test pass. The focus here is on getting the test to succeed, rather than on writing perfect or optimized code. 3. **Refactor**: Once the test passes, the developer refactors the code to improve its structure and readability while ensuring that all tests still pass. This step is crucial for maintaining code quality and manageability. The importance of TDD in software development lies in its ability to enhance code quality, reduce bugs, and improve the design of the software. By writing tests first, developers are encouraged to think through the requirements and design before implementation. This 

In [4]:
# Run once in a fresh kernel, then restart the kernel before running the model cell.
# A CUDA-enabled PyTorch environment is recommended for Qwen3.5-4B.
import sys
!{sys.executable} -m pip install -q torch transformers accelerate bitsandbytes "typing_extensions>=4.12"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 22.8 MB/s eta 0:00:00


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_feedback_model():
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {
        "device_map": "auto",
        "torch_dtype": torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        "trust_remote_code": True,
    }
    if LOAD_IN_4BIT and torch.cuda.is_available():
        from transformers import BitsAndBytesConfig

        model_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
    model.eval()
    return tokenizer, model


print(f"CUDA available: {torch.cuda.is_available()}")
tokenizer, model = load_feedback_model()
print("Model loaded successfully.")

CUDA available: True


config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

... (4 outputs left out)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/427 [00:00<?, ?it/s]

Model loaded successfully.


In [6]:
def load_completed_records(path: Path) -> dict[int, dict]:
    if not path.exists():
        return {}
    records = {}
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            if line.strip():
                record = json.loads(line)
                records[int(record["row_index"])] = record
    return records


def generate_one(row: pd.Series) -> str:
    encoded = tokenizer.apply_chat_template(
        build_messages(row),
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        enable_thinking=False,
    )
    if hasattr(encoded, "input_ids"):
        inputs = {
            key: value.to(model.device)
            for key, value in encoded.items()
            if hasattr(value, "to")
        }
        input_length = inputs["input_ids"].shape[-1]
    else:
        inputs = {"input_ids": encoded.to(model.device)}
        input_length = encoded.shape[-1]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    generated_tokens = output[0, input_length:]
    feedback = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    # Keep each feedback value on one physical CSV line.
    return " ".join(feedback.split())


def run_feedback(limit: int | None = None, force: bool = False, random_state: int | None = None,) -> pd.DataFrame:
    completed = {} if force else load_completed_records(RAW_OUTPUT_PATH)

    if limit is None:
        rows = parsed_df
    else:
        if limit < 1:
            raise ValueError("limit must be at least 1")
        rows = parsed_df.sample(n=min(limit, len(parsed_df)), random_state=random_state)

    records = dict(completed)

    with RAW_OUTPUT_PATH.open("a", encoding="utf-8") as handle:
        for row_index, row in rows.iterrows():
            existing = records.get(int(row_index))
            if existing and existing.get("status") == "ok" and existing.get("feedback", "").strip():
                continue

            started = time.perf_counter()
            try:
                feedback = generate_one(row)
                status = "ok" if feedback else "empty"
                error = None if feedback else "Model returned empty feedback"
            except Exception as exc:
                feedback = ""
                status = "error"
                error = f"{type(exc).__name__}: {exc}"

            record = {
                "row_index": int(row_index),
                "status": status,
                "feedback": feedback,
                "error": error,
                "elapsed_seconds": round(time.perf_counter() - started, 3),
            }
            handle.write(json.dumps(record, ensure_ascii=True) + "\n")
            handle.flush()
            records[int(row_index)] = record
            print(f"{row_index + 1}/{len(rows)} status={status}")

    missing = [int(index) for index in rows.index if int(index) not in records]
    if missing:
        raise RuntimeError(f"Missing generation records for rows: {missing[:10]}")

    feedback_values = [records[int(index)].get("feedback", "") for index in rows.index]
    result = raw_df.iloc[rows.index].copy()
    result["feedback"] = feedback_values
    result.to_csv(OUTPUT_PATH, index=False, quoting=csv.QUOTE_MINIMAL)
    return result



def validate_output(result: pd.DataFrame,limit: int | None = None) -> None:
    expected_columns = EXPECTED_COLUMNS + ["feedback"]
    if list(result.columns) != expected_columns:
        raise AssertionError(f"Unexpected output columns: {list(result.columns)}")
    if len(result) != (len(parsed_df) if limit is None else min(limit, len(parsed_df))):
        raise AssertionError(f"Expected {(len(parsed_df) if limit is None else min(limit, len(parsed_df)))} rows, got {len(result)}")
    if result["feedback"].fillna("").str.strip().eq("").any():
        print("Warning: one or more rows have empty feedback; inspect the JSONL errors.")
    print(f"Output written to {OUTPUT_PATH}")
    print(f"Raw records written to {RAW_OUTPUT_PATH}")

In [7]:
# Smoke test: generate a single row first. Set RUN_LIMIT = None for the full dataset.
RUN_LIMIT = 1
smoke_result = run_feedback(limit=RUN_LIMIT, force=False)
display(smoke_result[["feedback"]])

174/1 status=ok


,feedback
173,You've got the core definition spot-on: you co...


In [8]:
# Full run: execute this cell after the smoke test succeeds.
RUN_LIMIT = 50
full_result = run_feedback(limit=RUN_LIMIT, force=True, random_state=42)
validate_output(full_result, limit=RUN_LIMIT)
display(full_result[["feedback"]].head())

469/50 status=ok
297/50 status=ok
4615/50 status=ok
4884/50 status=ok
4285/50 status=ok
3378/50 status=ok
133/50 status=ok
849/50 status=ok
1184/50 status=ok
1118/50 status=ok
2099/50 status=ok
3930/50 status=ok
1322/50 status=ok
1698/50 status=ok
3048/50 status=ok
2518/50 status=ok
2719/50 status=ok
4975/50 status=ok
3246/50 status=ok
24/50 status=ok
4622/50 status=ok
539/50 status=ok
80/50 status=ok
2852/50 status=ok
3362/50 status=ok
752/50 status=ok
3465/50 status=ok
2544/50 status=ok
3998/50 status=ok
2679/50 status=ok
5063/50 status=ok
654/50 status=ok
2223/50 status=ok
1504/50 status=ok
3260/50 status=ok
3600/50 status=ok
1345/50 status=ok
1216/50 status=ok
1761/50 status=ok
2230/50 status=ok
1145/50 status=ok
152/50 status=ok
578/50 status=ok
5164/50 status=ok
3514/50 status=ok
3502/50 status=ok
4024/50 status=ok
502/50 status=ok
3671/50 status=ok
2941/50 status=ok
Output written to /content/sample_data/training_data_mol_with_feedback.csv
Raw records written to /content/sample_

,feedback
468,You've got the core TDD cycle down perfectly! ...
296,You've got the right spirit here by focusing o...
4614,You've captured the core purpose of the struct...
4883,You've got the part about whitespace right—rea...
4284,You've got this part right by correctly identi...
